# geobr DuckDB demo

Requires: `pip install geobr[duckdb]`

Each geobr snapshot is registered as an independent view (`schools_2020`, `biomes_2019`, ...). Use `query()` to run SQL across snapshots — missing views are downloaded automatically.

In [ ]:
# 1) Basic query + filter (auto-downloads states_2020 on first use)
from geobr import query

query(
    """
    SELECT name_state, abbrev_state
    FROM states_2020
    WHERE abbrev_state = 'RJ'
    """
).df()

In [ ]:
# 2) Geo functions via query()
from geobr import query

query(
    """
    SELECT name_state, ST_Area(geometry) / 1e6 AS area_km2
    FROM states_2020
    ORDER BY area_km2 DESC
    LIMIT 5
    """
).df()

In [ ]:
# 3) Cross-dataset spatial join (auto-resolver downloads missing snapshots)
query(
    """
    SELECT count(*) AS schools_in_amazon
    FROM schools_2020 AS s
    JOIN biomes_2019 AS b
      ON ST_Within(s.geometry, b.geometry)
    WHERE b.name_biome ILIKE '%Amaz%'
    """
).df()

In [ ]:
# 4) Round-trip to GeoPandas
from geobr import to_geopandas

gdf = to_geopandas("states_2020")
gdf.plot(column="name_state", legend=False, figsize=(6, 6))